# pyQuil Tutorial: Running on a Real QPU (requires QCS access)

A tour of pyQuil's QCS-only features: running on a live quantum processor, job
cancellation, reading a real device's topology/ISA, rewiring strategies on real hardware,
and alternative QPU endpoints. These are the counterparts to
**`1_simulation.ipynb`** (which only needs the local QVM/`quilc`) that require an actual
Rigetti QCS account.

**Prerequisites:**
- A Rigetti QCS account with a reservation/engagement on a quantum processor.
- QCS credentials configured in this environment (e.g. `qcs auth login`, or the
  `~/.qcs/settings.toml` / `~/.qcs/secrets.toml` files).
- `quilc` still runs locally for Quil-to-native-Quil translation, so start it as usual
  (the cell below checks); only the final executable-translation and execution steps go
  through QCS.

> **Note:** this environment currently has **no QCS credentials configured**, so the
> cells below were executed as-is and show the real connection/authentication errors
> you'd get without access. Once you have a QCS account and reservation, set `QPU_NAME`
> below to your processor's name (see your QCS dashboard or `list_quantum_computers()`)
> and re-run.

In [1]:
!tail -n 3 quilc.log qvm.log

==> quilc.log <==
<134>1 2026-11-03T04:28:12Z r002.ib.bridges2.psc.edu quilc 34614 LOG0002 [rigetti@0000 methodName="quil_to_native_quil" requestID="b5f5505e-1f34-461b-87a7-473d45e1fa6a" wallTime="0.015" error="false"] Requested quil_to_native_quil completed
<134>1 2026-11-03T04:28:13Z r002.ib.bridges2.psc.edu quilc 34614 - - Request e8072001-8739-4512-abd1-f97397257f25 received for get_version_info
<134>1 2026-11-03T04:28:13Z r002.ib.bridges2.psc.edu quilc 34614 LOG0002 [rigetti@0000 methodName="get_version_info" requestID="e8072001-8739-4512-abd1-f97397257f25" wallTime="0.0" error="false"] Requested get_version_info completed

==> qvm.log <==
<135>1 2026-11-03T04:28:13Z r002.ib.bridges2.psc.edu qvm 34621 - - [127.0.0.1 Session:1181] Finished in 17 ms
<134>1 2026-11-03T04:28:13Z r002.ib.bridges2.psc.edu qvm 34621 - - 127.0.0.1 - [2026-11-02 23:28:13] "POST / HTTP/1.1" 200 6014 "-" "-"



In [2]:
QPU_NAME = "Aspen-M-3"  # replace with your QCS quantum processor name

## Part 1: Running on a QPU

In [3]:
import warnings
from pyquil._deprecation import PyQuilDeprecationWarning
# Hide pyQuil 4.22's notices that the QVM is deprecated in favour of pyQuil v5
# (the QVM still works; see 0_Jupyter_Pykernel_from_Container_image.ipynb).
warnings.filterwarnings("ignore", category=PyQuilDeprecationWarning)

from pyquil import get_qc

# Get a QPU
qpu = get_qc(QPU_NAME)

# Get a QVM with the same topology as the QPU
qvm = get_qc(QPU_NAME, as_qvm=True)


GetISAError: Call failed during http request: error in refresh_qcs_token: No refresh token is configured within the selected QCS credential.

With `as_qvm=True` you get a QVM with the same topology as the named QPU. For noise models, see the *Noise and quantum computation* notebook.

### Differences between a QVM- and a QPU-based `QuantumComputer`

For QVM targets, `qc.qam` is a `QVM` and `qc.compiler` is a `QVMCompiler`. For QPU targets, `qc.qam` is a `QPU` and `qc.compiler` is a `QPUCompiler`. They share common interfaces (`QAM` and `AbstractCompiler`), but each may have backend-specific methods. Use type checks to access those safely:

#### Requesting a job cancellation

While a job is pending on a QPU, you can cancel it with `cancel` (not available on the QVM):

In [4]:
from pyquil.api import QPU
from pyquil import Program
from pyquil.gates import X, MEASURE

p = Program()
ro = p.declare('ro', 'BIT', 1)
p += X(0)
p += MEASURE(0, ro[0])

job_handle = qpu.qam.execute(qpu.compile(p))

if isinstance(qpu.qam, QPU):
    try:
        qpu.qam.cancel(job_handle)
        print("Job was cancelled")
    except Exception:  # the docs catch QpuApiError, raised by the QCS SDK
        # If this error was raised, then the job failed to be cancelled.
        result = qpu.qam.get_result(job_handle)
else:
    print("QVM target: cancellation not supported")


NameError: name 'qpu' is not defined

### Providing your own quantum processor topology

You can define a custom topology by specifying qubits and edges:

In [5]:
import networkx as nx
from pyquil import get_qc
from pyquil.quantum_processor import NxQuantumProcessor
from pyquil.noise import decoherence_noise_with_asymmetric_ro

qpu = get_qc(QPU_NAME)
isa = qpu.to_compiler_isa()
qubits = sorted(int(k) for k in isa.qubits.keys())[:16]
edges = [(q1, q2) for q1 in qubits for q2 in qubits if f"{q1}-{q2}" in isa.edges]

# Build the NX graph
topo = nx.from_edgelist(edges)
# You would uncomment the next line if you have disconnected qubits
# topo.add_nodes_from(qubits)
quantum_processor = NxQuantumProcessor(topo)
quantum_processor.noise_model = decoherence_noise_with_asymmetric_ro(quantum_processor.to_compiler_isa())  # Optional


GetISAError: Call failed during http request: error in refresh_qcs_token: No refresh token is configured within the selected QCS credential.

## Part 2: Rewiring Strategies on a Real Device

> **Note:** A SWAP costs three CZ or three XY gates. Devices that support both can build a SWAP from two gates (one CZ and one XY).

### Initial rewiring

Control how the initial mapping is built with `PRAGMA INITIAL_REWIRING`:

```
# <type> can be NAIVE, RANDOM, PARTIAL, or GREEDY
#
# The double quotes are required.
PRAGMA INITIAL_REWIRING "<type>"
```

### The default initial rewiring strategy

Without `INITIAL_REWIRING`, the compiler picks:

- **`NAIVE`** when every instruction's qubits already satisfy the device topology
- **`PARTIAL`** otherwise

The docs illustrate these strategies on a real device topology (`Aspen-X`), which requires QCS access. The cells below use it as in the docs; the final cell reproduces the idea locally on the 3-qubit line.

In [6]:
# Requires QCS access. Neighboring qubits -> NAIVE is applied, output: CZ 3 4
from pyquil import Program, get_qc
from pyquil.gates import CZ

qc_aspen = get_qc(QPU_NAME, as_qvm=True)
p = Program(CZ(3, 4))
print(qc_aspen.compile(p))

GetISAError: Call failed during http request: error in refresh_qcs_token: No refresh token is configured within the selected QCS credential.

#### NAIVE

Logical qubit `i` maps directly to physical qubit `i`. The compiler generally will **not** move an instruction's qubits, even if that results in poor fidelity.

In [7]:
# Requires QCS access. A low-fidelity CZ is kept as-is: output CZ 0 1
p = Program('PRAGMA INITIAL_REWIRING "NAIVE"', CZ(0, 1))
print(qc_aspen.compile(p))

NameError: name 'qc_aspen' is not defined

In [8]:
# Requires QCS access. Non-neighboring qubits need a virtual swap: docs output CZ 6 5
p = Program('PRAGMA INITIAL_REWIRING "NAIVE"', CZ(0, 2))
print(qc_aspen.compile(p))

NameError: name 'qc_aspen' is not defined

> **Note:** When NAIVE mode has to rewire, the result isn't guaranteed to have optimal fidelity.

#### PARTIAL

The compiler starts with an empty logical-to-physical mapping and fills it in as it compiles, giving it the freedom to choose a mapping that optimizes the program's fidelity.

In [9]:
# Requires QCS access. The compiler moves the CZ to a better pair: docs output CZ 20 27
p = Program('PRAGMA INITIAL_REWIRING "PARTIAL"', CZ(0, 1))
print(qc_aspen.compile(p))

NameError: name 'qc_aspen' is not defined

#### GREEDY

The compiler builds the initial mapping from a greedy optimization of distances between the program's qubits and the device's. It's faster than PARTIAL but gives lower-fidelity programs — use it when compile speed matters more than fidelity.

#### Which strategy should I use?

| Desired effect | Recommended strategy |
|---|---|
| Maximize program execution fidelity | `PARTIAL` |
| Keep the input program's qubits where possible | `NAIVE` |
| Faster qubit allocation at the expense of fidelity | `GREEDY` |

> **Note:** Each strategy has the drawbacks described above.

### Try it locally: compare strategies on the line topology

## Part 3: Alternative QPU Endpoints

## Alternative QPU endpoints

Rigetti QCS supports alternative endpoints for QPU access — useful, for example, for mock endpoints that simulate execution without an active reservation. Pass an `endpoint_id` wherever a `quantum_processor_id` goes. Compilation behaves the same as with the default endpoint, but execution goes through the alternate service, so don't treat results as meaningful.

In [10]:
from pyquil import get_qc
from pyquil.api import QPU

# Option 1
qc = get_qc(QPU_NAME, endpoint_id="my_endpoint")

# Option 2
qam = QPU(quantum_processor_id=QPU_NAME, endpoint_id="my_endpoint")


GetISAError: Call failed during http request: error in refresh_qcs_token: No refresh token is configured within the selected QCS credential.

### Kill the servers after use in **0_Jupyter_Pykernel_from_Container_image** notebook